In [0]:
%run ./00-Config

In [0]:
from pyspark.sql import functions as F

In [0]:
# ==========================================
# CONFIGURAÇÕES
# ==========================================
# ALTERADO:
# todos os paths agora vêm do notebook config

schema_location = (
    f"{config_volume_path}/schemas/bronze"
)

checkpoint_path = (
    f"{config_volume_path}/checkpoints/bronze"
)

landing_path = landing_volume_path

# ALTERADO:
# bronze agora escreve na tabela criada no config

bronze_table_name = (
    f"{catalog_name}.bronze.openbrewery_bronze"
)

In [0]:
# COMMAND ----------

# ==========================================
# LEITURA STREAMING COM AUTO LOADER
# ==========================================

raw_stream_df = (

    spark.readStream

    .format("cloudFiles")

    .option(
        "cloudFiles.format",
        "json"
    )

    .option(
        "cloudFiles.schemaLocation",
        schema_location
    )

    .option(
        "cloudFiles.inferColumnTypes",
        "true"
    )

    .option(
        "cloudFiles.schemaEvolutionMode",
        "addNewColumns"
    )

    .option(
        "cloudFiles.rescuedDataColumn",
        "_rescued_data"
    )

    .option(
        "multiLine",
        "true"
    )

    .load(landing_path)
)


In [0]:
# COMMAND ----------

# ==========================================
# EXPLODE DO ARRAY DATA
# ==========================================

bronze_df = (

    raw_stream_df

    .withColumn(
        "brewery",
        F.explode("data")
    )

    .select(

        # ==========================================
        # METADATA ORIGINAL
        # ==========================================

        "metadata",

        # ==========================================
        # RESCUED DATA
        # ==========================================

        "_rescued_data",

        # ==========================================
        # CAMPOS DE NEGÓCIO
        # ==========================================

        F.col("brewery.id").alias("id"),

        F.col("brewery.name").alias("name"),

        F.col("brewery.brewery_type").alias(
            "brewery_type"
        ),

        F.col("brewery.address_1").alias(
            "address_1"
        ),

        F.col("brewery.address_2").alias(
            "address_2"
        ),

        F.col("brewery.address_3").alias(
            "address_3"
        ),

        F.col("brewery.city").alias(
            "city"
        ),

        F.col("brewery.state").alias(
            "state"
        ),

        F.col("brewery.state_province").alias(
            "state_province"
        ),

        F.col("brewery.postal_code").alias(
            "postal_code"
        ),

        F.col("brewery.country").alias(
            "country"
        ),

        F.col("brewery.longitude").alias(
            "longitude"
        ),

        F.col("brewery.latitude").alias(
            "latitude"
        ),

        F.col("brewery.phone").alias(
            "phone"
        ),

        F.col("brewery.website_url").alias(
            "website_url"
        ),

        F.col("brewery.street").alias(
            "street"
        ),

        # ==========================================
        # METADADOS TÉCNICOS
        # ==========================================

        F.col("_metadata.file_name").alias(
            "source_file"
        ),

        F.col("_metadata.file_path").alias(
            "source_path"
        ),

        F.col("_metadata.file_modification_time").alias(
            "file_modification_time"
        ),

        F.col("_metadata.file_size").alias(
            "file_size"
        )
    )
)

In [0]:
# COMMAND ----------

# ==========================================
# METADADOS TÉCNICOS
# ==========================================

bronze_df = (

    bronze_df

    .withColumn(
        "ingestion_timestamp",
        F.current_timestamp()
    )

    .withColumn(
        "ingestion_date",
        F.to_date(
            F.current_timestamp()
        )
    )

    .withColumn(
        "execution_id",
        F.date_format(
            F.current_timestamp(),
            "yyyyMMddHHmmss"
        )
    )

    # ==========================================
    # HASH PARA DETECÇÃO DE MUDANÇA
    # ==========================================

    .withColumn(

        "record_hash",

        F.sha2(

            F.concat_ws(

                "||",

                F.col("id"),

                F.col("name"),

                F.col("city"),

                F.col("state"),

                F.col("country")
            ),

            256
        )
    )
)

In [0]:
# COMMAND ----------

# ==========================================
# DEDUPLICAÇÃO
# ==========================================
# ALTERADO:
# mantém apenas um registro por id
# dentro do microbatch

bronze_df = bronze_df.dropDuplicates(
    ["id"]
)

In [0]:

# ==========================================
# ESCRITA STREAMING PARA DELTA
# ==========================================
# ALTERADO:
# agora escreve diretamente na tabela UC
# criada no notebook config

query = (

    bronze_df.writeStream

    .format("delta")

    .outputMode("append")

    .option(
        "checkpointLocation",
        checkpoint_path
    )

    .option(
        "mergeSchema",
        "true"
    )

    .trigger(
        availableNow=True
    )

    .toTable(
        bronze_table_name
    )
)


In [0]:
# COMMAND ----------

# ==========================================
# EXECUÇÃO
# ==========================================

query.awaitTermination()

print("=" * 60)
print("BRONZE LOAD FINISHED")
print(f"Table: {bronze_table_name}")
print("=" * 60)